# LAB-2: Sistema Avançado de Sensores Radar

Neste laboratório, expandiremos o sistema de raycasting do LAB-1 para incluir:
- **8 sensores** distribuídos em 360°
- **Visualização de radar** circular
- **Histograma de distâncias**
- **Detecção de padrões** de obstáculos

In [ ]:
# CÓDIGO LAB-2: Sistema de Sensores Radar com 8 Raios
import pygame
import math
import numpy as np

LARGURA, ALTURA = 1000, 750
FPS = 60
COR_FUNDO = (20, 24, 30)
COR_ROBO = (0, 200, 255)
COR_OBSTACULO = (180, 50, 50)
COR_RAIO_LIVRE = (0, 255, 100)
COR_RAIO_COLISAO = (255, 200, 0)
COR_RADAR = (0, 100, 150)

class AdvancedRaycastRobot:
    """Robô com 8 sensores distribuídos em 360 graus."""
    
    def __init__(self, x, y, theta=0.0, num_sensores=8):
        self.x = float(x)
        self.y = float(y)
        self.theta = float(theta)
        self.num_sensores = num_sensores
        
        # Distribuição uniforme de sensores em 360 graus
        self.sensor_angles = [2 * math.pi * i / num_sensores for i in range(num_sensores)]
        self.sensor_range = 150.0
        self.sensor_readings = [self.sensor_range] * num_sensores
        
        # Histórico para análise
        self.history = []
        self.max_history = 100

    def cast_rays(self, obstacles):
        """Verifica interseção dos raios com obstáculos retangulares."""
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            dx = math.cos(angle)
            dy = math.sin(angle)
            
            min_distance = self.sensor_range
            
            for obs in obstacles:
                distance = self._ray_rect_intersection(self.x, self.y, dx, dy, obs)
                if distance is not None and distance < min_distance:
                    min_distance = distance
            
            self.sensor_readings.append(min_distance)
        
        # Armazena no histórico
        self.history.append(self.sensor_readings.copy())
        if len(self.history) > self.max_history:
            self.history.pop(0)

    def _ray_rect_intersection(self, x0, y0, dx, dy, rect):
        """Calcula interseção de um raio com um retângulo (AABB)."""
        rx, ry, rw, rh = rect
        x_min, x_max = rx, rx + rw
        y_min, y_max = ry, ry + rh
        
        t_min = float('-inf')
        t_max = float('inf')
        
        # Verificação em X
        if abs(dx) > 1e-6:
            t1 = (x_min - x0) / dx
            t2 = (x_max - x0) / dx
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif x0 < x_min or x0 > x_max:
            return None
        
        # Verificação em Y
        if abs(dy) > 1e-6:
            t1 = (y_min - y0) / dy
            t2 = (y_max - y0) / dy
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif y0 < y_min or y0 > y_max:
            return None
        
        if t_min < t_max and t_min > 0 and t_min < self.sensor_range:
            return t_min
        
        return None

    def get_sensor_stats(self):
        """Retorna estatísticas dos sensores."""
        if not self.sensor_readings:
            return {'min': 0, 'max': 0, 'media': 0, 'obstaculos': 0}
        
        obstaculos = sum(1 for d in self.sensor_readings if d < self.sensor_range - 0.1)
        return {
            'min': min(self.sensor_readings),
            'max': max(self.sensor_readings),
            'media': np.mean(self.sensor_readings),
            'obstaculos': obstaculos
        }

    def draw(self, screen):
        """Desenha o robô, sensores e radar."""
        # Desenha o corpo
        pygame.draw.circle(screen, COR_ROBO, (int(self.x), int(self.y)), 8)
        pygame.draw.line(screen, COR_ROBO, (self.x, self.y), 
                         (self.x + 12 * math.cos(self.theta), 
                          self.y + 12 * math.sin(self.theta)), 2)
        
        # Desenha os raios dos sensores
        for i, beta in enumerate(self.sensor_angles):
            angle = self.theta + beta
            distance = self.sensor_readings[i]
            end_x = self.x + distance * math.cos(angle)
            end_y = self.y + distance * math.sin(angle)
            
            cor = COR_RAIO_COLISAO if distance < self.sensor_range - 0.1 else COR_RAIO_LIVRE
            pygame.draw.line(screen, cor, (self.x, self.y), (end_x, end_y), 1)

    def draw_radar(self, screen, x, y, raio=100):
        """Desenha o radar circular dos sensores."""
        # Círculo externo
        pygame.draw.circle(screen, COR_RADAR, (x, y), raio, 1)
        pygame.draw.circle(screen, COR_RADAR, (x, y), raio // 2, 1)
        
        # Linhas de eixo
        pygame.draw.line(screen, COR_RADAR, (x - raio, y), (x + raio, y))
        pygame.draw.line(screen, COR_RADAR, (x, y - raio), (x, y + raio))
        
        # Plotar sensores como pontos no radar
        for i, distance in enumerate(self.sensor_readings):
            angle = i * 2 * math.pi / self.num_sensores
            # Normaliza distância para o raio
            r = (distance / self.sensor_range) * raio
            px = x + r * math.cos(angle)
            py = y + r * math.sin(angle)
            
            cor = COR_RAIO_COLISAO if distance < self.sensor_range - 0.1 else COR_RAIO_LIVRE
            pygame.draw.circle(screen, cor, (int(px), int(py)), 3)

    def update(self, keys, obstacles):
        """Atualiza posição e sensores."""
        if keys[pygame.K_w] or keys[pygame.K_UP]:
            self.x += 3 * math.cos(self.theta)
            self.y += 3 * math.sin(self.theta)
        if keys[pygame.K_s] or keys[pygame.K_DOWN]:
            self.x -= 2 * math.cos(self.theta)
            self.y -= 2 * math.sin(self.theta)
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:
            self.theta -= 0.05
        if keys[pygame.K_d] or keys[pygame.K_RIGHT]:
            self.theta += 0.05
        
        self.x = max(0, min(LARGURA - 250, self.x))
        self.y = max(0, min(ALTURA, self.y))
        
        self.cast_rays(obstacles)


def draw_obstacles(screen, obstacles):
    """Desenha obstáculos na tela."""
    for obs in obstacles:
        x, y, w, h = obs
        pygame.draw.rect(screen, COR_OBSTACULO, (x, y, w, h))


def draw_histogram(screen, robot, x, y, width=200, height=100):
    """Desenha histograma das leituras dos sensores."""
    # Moldura
    pygame.draw.rect(screen, (100, 100, 100), (x, y, width, height), 1)
    
    if not robot.sensor_readings:
        return
    
    bar_width = width / len(robot.sensor_readings)
    
    for i, distance in enumerate(robot.sensor_readings):
        # Normaliza altura
        bar_height = (distance / robot.sensor_range) * height
        bar_x = x + i * bar_width
        bar_y = y + height - bar_height
        
        cor = COR_RAIO_COLISAO if distance < robot.sensor_range - 0.1 else COR_RAIO_LIVRE
        pygame.draw.rect(screen, cor, (bar_x, bar_y, bar_width - 1, bar_height))


def main():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB-2: Sistema Radar com 8 Sensores")
    clock = pygame.time.Clock()
    
    obstacles = [
        (100, 100, 150, 30),
        (400, 150, 30, 200),
        (650, 400, 150, 30),
        (300, 450, 250, 30),
        (50, 350, 30, 150),
    ]
    
    robot = AdvancedRaycastRobot(LARGURA // 3, ALTURA // 2, num_sensores=8)
    
    running = True
    while running:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
        
        keys = pygame.key.get_pressed()
        robot.update(keys, obstacles)
        
        # Renderiza
        screen.fill(COR_FUNDO)
        draw_obstacles(screen, obstacles)
        robot.draw(screen)
        
        # Radar
        robot.draw_radar(screen, LARGURA - 150, 100, raio=100)
        
        # Histograma
        draw_histogram(screen, robot, LARGURA - 250, ALTURA - 150)
        
        # HUD
        font = pygame.font.Font(None, 20)
        stats = robot.get_sensor_stats()
        
        info_texts = [
            f"Posição: ({robot.x:.0f}, {robot.y:.0f})",
            f"Orientação: {math.degrees(robot.theta):.1f}°",
            f"Sensores: {robot.num_sensores}",
            f"Mín/Máx/Média: {stats['min']:.0f}/{stats['max']:.0f}/{stats['media']:.0f}",
            f"Obstáculos detectados: {stats['obstaculos']}"
        ]
        
        for i, text in enumerate(info_texts):
            surface = font.render(text, True, (200, 200, 200))
            screen.blit(surface, (10, 10 + i * 25))
        
        # Instruções
        inst_font = pygame.font.Font(None, 16)
        inst = inst_font.render("W/Seta-Cima: Avançar | S/Seta-Baixo: Recuar | A/Esq: Girar | D/Dir: Girar | ESC: Sair", True, (150, 150, 150))
        screen.blit(inst, (10, ALTURA - 25))
        
        pygame.display.flip()
    
    pygame.quit()

if __name__ == "__main__":
    main()